In [0]:
data = spark.read.table("`01_bronze`.raw.customers")

In [0]:
import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("birthday", data)
display(data.select(F.count("*")))



## Handling null values in the state column

In [0]:
def state_null_handle(df):

    mapping_df = df.filter(F.col("state").isNotNull()) \
               .select("state_code", "state") \
               .dropDuplicates(["state_code"])
    return df.alias("a").join(
        mapping_df.alias("b"), 
        on="state_code", 
        how="left"
    ).select(
        F.col("a.customerkey"),
        F.col("a.gender"),
        F.col("a.name"),
        F.col("a.city"),
        F.col("a.state_code"),
        F.coalesce(F.col("a.state"), F.col("b.state")).alias("state"),
        F.col("a.zip_code"),
        F.col("a.country"),
        F.col("a.continent"),
        F.col("a.birthday")
    )

data = state_null_handle(data)
display(data.select(F.count("*")))


## Handling zip code column null values

In [0]:
def zip_code_null_handle(df):
    mapping_df = df.filter(F.col("zip_code").isNotNull())\
                   .select("city", "state", "zip_code")\
                   .dropDuplicates(["city", "state"])

    return df.alias("a").join(
        mapping_df.alias("b"), 
        on=["city", "state"], 
        how="left"
    ).select(
        "a.customerkey",
        "a.gender",
        "a.name",
        "a.city",
        "a.state_code",
        "a.state",
        F.coalesce(F.col("a.zip_code"), F.col("b.zip_code")).alias("zip_code"),
        "a.country",
        "a.continent",
        "a.birthday"
    )
data = zip_code_null_handle(data)
display(data.select(F.count("*")))

In [0]:
def dataTypeHandle(df,colname,dataType):
    if dict(df.dtypes)[colname] == 'string':
        return df.withColumn(colname,F.trim(F.col(colname)).try_cast(dataType))
    if dict(df.dtypes)[colname] == dataType:
        return df.withColumn(colname,F.col(colname))


data = dataTypeHandle(data,'customerkey','int')
data = dataTypeHandle(data,'gender','string')
data = dataTypeHandle(data,'name','string')
data = dataTypeHandle(data,'city','string')
data = dataTypeHandle(data,'state_code','string')
data = dataTypeHandle(data,'state','string')
data = dataTypeHandle(data,'zip_code','string')
data = dataTypeHandle(data,'country','string')
data = dataTypeHandle(data,'continent','string')
data = dataTypeHandle(data,'birthday','date')
data = data.filter(F.col("customerkey").isNotNull())

data.write.mode("overwrite").saveAsTable("02_silver.transformation.customers")

In [0]:
from pyspark.sql.functions import count,when,col

temp = spark.read.table("02_silver.transformation.customers")
null_counts = temp.select([count(when(col(c).isNull(),c)).alias(c) for c in temp.columns])
display(null_counts)